In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from mlforecast import MLForecast
from tinyshift.modelling import TwoStageForecasterWrapper
from tinyshift.series import economic_loss, tail_risk
import lightgbm as lgb
from mlforecast.lag_transforms import RollingMean, RollingStd

In [2]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

In [3]:
# Financial parameters
selling_price = 320      # Unit selling price
unit_cost = 90         # Unit acquisition cost
annual_holding_rate = 0.14 # Annual capital holding rate (12% p.a.)

days_obsoletes = 180

# 1. Underage Cost (Cu): Lost margin per unfulfilled unit
c_u = selling_price - unit_cost

# 2. Overage Cost (Co): Holding cost + daily obsolescence rate
daily_obsolescence_cost = unit_cost / days_obsoletes
daily_holding_cost = (unit_cost * annual_holding_rate) / 365

estimated_holding_days = 30
c_o = (daily_holding_cost + daily_obsolescence_cost) * estimated_holding_days

In [4]:
fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='rmse',
            n_estimators=100,
            learning_rate=0.05,
            random_state=42,
            verbosity=-1
        )
    },
    freq='MS',
    lags=[1, 2, 7],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [5]:
issm = TwoStageForecasterWrapper(fcst)

In [6]:
issm.fit(train, static_features=[])

In [7]:
test = test.copy()
test.loc[:,"cu"] = 1
test.loc[:,"co"] = 2

In [8]:
pred = issm.optimize(h=12, underage_cost="cu", overage_cost="co", X_df=test).rename(columns={"y_optimal": "ISSMWrapper"})

In [9]:
pred= pd.merge(pred, test, on=["unique_id", "ds"])

In [10]:
economic_loss(pred, ["ISSMWrapper"], id_col="unique_id")

,unique_id,metric,ISSMWrapper
0,1,economic_loss,971


In [11]:
tail_risk(pred, ["ISSMWrapper"], id_col="unique_id")

,unique_id,metric,ISSMWrapper
0,1,expected_cost,80.92
1,1,std_dev,72.26
2,1,var95,214.00
3,1,cvar95,214.00
4,1,worst_scenario,214.00
